# 00 · Common definitions

Shared paths, data loading, screening, features, models and metrics. Every other
notebook loads this one with `%run ./00_common.ipynb`, so each definition exists
exactly once. Running this notebook on its own only defines functions.

In [ ]:
import os, io, pickle, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, wilcoxon

warnings.filterwarnings("ignore")

# repository root = the folder that holds data/metadata.csv
ROOT = Path.cwd()
while not (ROOT / "data" / "metadata.csv").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("data/metadata.csv not found above " + str(Path.cwd()))
    ROOT = ROOT.parent
RESULTS = ROOT / "results"; RESULTS.mkdir(exist_ok=True)
FIGS = ROOT / "figs"; FIGS.mkdir(exist_ok=True)
# raw per-cycle discharge waveforms, needed only by the SOC notebook
ORGANIZED = Path(os.environ.get("PCOE_ORGANIZED", ROOT / "organized"))
print("root:", ROOT)

## Protocol constants
Identical to the submitted paper.

In [ ]:
SEED = 42
SEEDS = (0, 1, 7, 42, 2024)
LOOKBACK = 10
EOL_SOH_PRIMARY = 80.0     # benchmark EOL threshold
EOL_SOH_CLASSIC = 70.0     # conventional threshold, reported as a censoring statistic
MIN_CYCLES = 40
MONOTONE_RHO = -0.8
INIT_SOH_MIN = 90.0
SUSTAIN = 3                # EOL = first cycle with smoothed SOH < thr for 3 consecutive cycles
LAMBDAS = [0.5, 0.7, 0.9]
ALPHA = 0.20               # alpha-lambda accuracy tolerance
CP_LEVEL = 0.90            # conformal target coverage
ALPHA_CP = 1 - CP_LEVEL
N_BOOT = 10000
IMPEDANCE = ["Re", "Rct"]

## Data loading
Per discharge cycle: capacity, nearest *preceding* impedance measurement
(`merge_asof` backward, so no future impedance leaks in), ambient temperature.
`soh_s` (centered median) defines the offline EOL label only; `soh_t` (trailing
median) is what the model features use.

In [ ]:
def load_cells():
    m = pd.read_csv(ROOT / "data" / "metadata.csv")
    d = m[m["type"] == "discharge"].copy()
    d["Capacity"] = pd.to_numeric(d["Capacity"], errors="coerce")
    d = d.dropna(subset=["Capacity"])
    d = d[d["Capacity"] > 0.1]                      # sensor dropouts / aborted cycles
    imp = m[m["type"] == "impedance"].copy()
    for c in ("Re", "Rct"):
        imp[c] = imp[c].astype(str).str.replace("(", "", regex=False).str.replace(")", "", regex=False)
        imp[c] = imp[c].str.split("+").str[0].str.split("-").str[0]
        imp[c] = pd.to_numeric(imp[c], errors="coerce")
    cells = {}
    for b, g in d.groupby("battery_id"):
        g = g.sort_values("uid").reset_index(drop=True)
        g["cycle"] = np.arange(1, len(g) + 1)
        n = len(g)
        qref = g["Capacity"].iloc[:max(5, int(0.25 * n))].max()
        g["soh"] = 100 * g["Capacity"] / qref
        g["soh_s"] = g["soh"].rolling(5, center=True, min_periods=1).median()   # label only
        g["soh_t"] = g["soh"].rolling(5, min_periods=1).median()                # causal, features
        gi = imp[imp["battery_id"] == b].sort_values("uid")
        g = g.drop(columns=["Re", "Rct"], errors="ignore")
        if len(gi):
            g = pd.merge_asof(g.sort_values("uid"), gi[["uid", "Re", "Rct"]], on="uid", direction="backward")
        else:
            g["Re"] = np.nan; g["Rct"] = np.nan
        g["temp"] = g["ambient_temperature"].astype(float)
        cells[b] = dict(df=g, n=n, qref=qref, temp=g["temp"].iloc[0])
    return cells

## Screening and censoring

In [ ]:
def find_eol(soh_s, cycles, thr):
    below = (soh_s < thr).to_numpy()
    for i in range(len(below) - SUSTAIN + 1):
        if below[i:i + SUSTAIN].all():
            return int(cycles.iloc[i])
    return None


def screen(cells, save=True):
    rows = []
    for b, c in cells.items():
        g = c["df"]
        rho = spearmanr(g["cycle"], g["soh_s"]).statistic if len(g) > 5 else 0.0
        init_ok = g["soh_s"].iloc[:5].mean() >= INIT_SOH_MIN
        long_ok = c["n"] >= MIN_CYCLES
        mono_ok = rho <= MONOTONE_RHO
        passes = long_ok and init_ok and mono_ok
        eol80 = find_eol(g["soh_s"], g["cycle"], EOL_SOH_PRIMARY) if passes else None
        eol70 = find_eol(g["soh_s"], g["cycle"], EOL_SOH_CLASSIC) if passes else None
        status = ("excluded: <%d cycles" % MIN_CYCLES if not long_ok else
                  "excluded: unreliable reference (initial SOH<%d%%)" % INIT_SOH_MIN if not init_ok else
                  "excluded: non-monotone fade (protocol-confounded)" if not mono_ok else
                  "eligible" if eol80 else "censored (never <%.0f%% SOH)" % EOL_SOH_PRIMARY)
        rows.append(dict(battery=b, n_cycles=c["n"], temp=c["temp"], q_ref=round(c["qref"], 3),
                         spearman_rho=round(float(rho), 3), eol80=eol80, eol70=eol70, status=status,
                         eligible=passes and eol80 is not None, censored=passes and eol80 is None))
        cells[b]["eol80"], cells[b]["eol70"] = eol80, eol70
    scr = pd.DataFrame(rows).sort_values("battery")
    if save:
        scr.to_csv(RESULTS / "screening.csv", index=False)
    return scr

## Features and labels
19 features per cycle. RUL label = EOL - k, using the centered-median EOL.

In [ ]:
def feature_table(cells, eligible, smooth="trailing", drop=()):
    col = {"trailing": "soh_t", "centered": "soh_s"}[smooth]
    frames = []
    for b in eligible:
        g = cells[b]["df"]
        eol = cells[b]["eol80"]
        s = g[col]
        feats = {"battery_id": b, "cycle": g["cycle"], "soh": s, "temp": g["temp"],
                 "q_ref": cells[b]["qref"], "Re": g["Re"], "Rct": g["Rct"]}
        for l in range(1, LOOKBACK + 1):
            feats[f"soh_lag{l}"] = s.shift(l)
        feats["slope10"] = s.diff().rolling(10, min_periods=5).mean()
        feats["slope25"] = s.diff().rolling(25, min_periods=10).mean()
        feats["std10"] = s.rolling(10, min_periods=5).std()
        feats["regen"] = (s.diff() > 0.5).cumsum()
        f = pd.DataFrame(feats)
        f["rul"] = eol - f["cycle"]
        f = f[f["cycle"] <= eol].dropna(subset=[f"soh_lag{LOOKBACK}"])
        frames.append(f)
    df = pd.concat(frames, ignore_index=True)
    feat_cols = [c for c in df.columns if c not in ("battery_id", "cycle", "rul") and c not in drop]
    return df, feat_cols


def xy(df, cols, med=None):
    X = df[cols]
    if med is None:
        med = X.median(numeric_only=True)          # imputation fitted on the training fold only
    return X.fillna(med).to_numpy(float), df["rul"].to_numpy(float), med

## Models
Fixed default hyperparameters, never tuned on held-out cells.

In [ ]:
def get_models(seed=SEED):
    from sklearn.linear_model import LinearRegression, Ridge
    from sklearn.ensemble import RandomForestRegressor
    import xgboost as xgb
    import lightgbm as lgb
    return {
        "Linear": LinearRegression(),
        "Ridge": Ridge(alpha=1.0, random_state=seed),
        "RandomForest": RandomForestRegressor(n_estimators=300, random_state=seed, n_jobs=-1),
        "XGBoost": xgb.XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.05,
                                    subsample=0.9, colsample_bytree=0.9, random_state=seed,
                                    verbosity=0, n_jobs=4),
        "LightGBM": lgb.LGBMRegressor(n_estimators=400, max_depth=4, learning_rate=0.05,
                                      subsample=0.9, colsample_bytree=0.9, random_state=seed,
                                      verbose=-1, n_jobs=4),
    }


HYPERPARAMS = [
    ("Linear", "ordinary least squares, no hyperparameters"),
    ("Ridge", "alpha=1.0"),
    ("RandomForest", "n_estimators=300, max_depth=None, min_samples_leaf=1, max_features=1.0"),
    ("XGBoost", "n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9"),
    ("LightGBM", "n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9"),
]

# candidate configurations for nested selection: (add cycle age k, predict RUL/k)
CONFIGS = {"base": (False, False), "age": (True, False), "ratio": (False, True), "age_ratio": (True, True)}


def fit_predict(name, train, test, cols, age, ratio, seed=SEED):
    cc = cols + (["age"] if age else [])
    Xtr, _, med = xy(train, cc)
    Xte, _, _ = xy(test, cc, med)
    ytr = train["rul"].to_numpy(float)
    k_tr, k_te = train["cycle"].to_numpy(float), test["cycle"].to_numpy(float)
    m = get_models(seed)[name]
    m.fit(Xtr, ytr / k_tr if ratio else ytr)
    p = m.predict(Xte)
    return np.clip(p * k_te if ratio else p, 0, None)

## Metrics
Error at life fractions lambda, EOL hit rate (+/-10 cycles), alpha-lambda accuracy
(alpha = 0.2), bootstrap-over-cells 95% CIs.

In [ ]:
def per_cell_errors(preds, cells, lambdas=LAMBDAS):
    rows = []
    for method, runs in preds.items():
        for b, r in runs.items():
            if r is None or len(r["cycle"]) == 0:
                continue
            eol = cells[b]["eol80"]
            for lmb in lambdas:
                k = round(lmb * eol)
                if k < r["cycle"].min() or k > r["cycle"].max():
                    continue                     # k outside the test segment -> not scored
                i = int(np.argmin(np.abs(r["cycle"] - k)))
                p, true = r["rul_pred"][i], eol - k
                rows.append(dict(method=method, battery=b, **{"lambda": lmb},
                                 err=p - true, abs_err=abs(p - true),
                                 hit=abs((k + p) - eol) <= 10,
                                 alpha_ok=abs(p - true) <= ALPHA * max(true, 1e-6)))
    return pd.DataFrame(rows)


def boot_ci(x, n=N_BOOT, seed=0):
    x = np.asarray(x, float)
    idx = np.random.default_rng(seed).integers(0, len(x), size=(n, len(x)))
    s = x[idx].mean(axis=1)
    return np.percentile(s, 2.5), np.percentile(s, 97.5)


def summarise(errs, tag):
    rows = []
    for (m, lmb), g in errs.groupby(["method", "lambda"]):
        lo, hi = boot_ci(g["abs_err"])
        hlo, hhi = boot_ci(g["hit"].astype(float))
        alo, ahi = boot_ci(g["alpha_ok"].astype(float))
        rows.append(dict(config=tag, method=m, **{"lambda": lmb}, n_cells=len(g),
                         MAE=g["abs_err"].mean(), MAE_sd=g["abs_err"].std(ddof=1),
                         MAE_ci_lo=lo, MAE_ci_hi=hi, RMSE=np.sqrt((g["err"] ** 2).mean()),
                         HR=g["hit"].mean(), HR_ci_lo=hlo, HR_ci_hi=hhi,
                         alpha_acc=g["alpha_ok"].mean(), alpha_ci_lo=alo, alpha_ci_hi=ahi))
    return pd.DataFrame(rows)


def cell_err(cells, b, cycles, pred):
    """mean |error| over lambda in {0.5, 0.7, 0.9} for one cell (inner-loop score)"""
    eol = cells[b]["eol80"]
    e = []
    for lmb in LAMBDAS:
        k = round(lmb * eol)
        if cycles.min() <= k <= cycles.max():
            i = int(np.argmin(np.abs(cycles - k)))
            e.append(abs(pred[i] - (eol - k)))
    return np.mean(e)


def wilcoxon_table(per_cell, lam=0.7):
    piv = per_cell[per_cell["lambda"] == lam].pivot(index="battery", columns="method", values="abs_err")
    rows, ms = [], list(piv.columns)
    for i, a in enumerate(ms):
        for b in ms[i + 1:]:
            d = piv[a] - piv[b]
            p = wilcoxon(piv[a], piv[b]).pvalue if (d != 0).any() else 1.0
            rows.append(dict(a=a, b=b, mean_a=piv[a].mean(), mean_b=piv[b].mean(), p=p))
    return pd.DataFrame(rows)

## Leave-one-battery-out with conformal intervals
Split conformal on 2 calibration cells (submitted protocol, now seeded per fold) and
cell-grouped CV+ (jackknife+, Barber et al. 2021) over all nine training cells.

In [ ]:
def conformal_fold(name, train, test, predict, seed, held, eol):
    """predict(name, train_df, test_df) -> RUL predictions. Returns rows + XGB band frame."""
    yte, cyc = test["rul"].to_numpy(float), test["cycle"].to_numpy()
    trc = sorted(train["battery_id"].unique())
    rng = np.random.default_rng(seed * 1000 + int(held[1:]))
    cal_cells = list(rng.choice(trc, size=2, replace=False))
    proper, cal = train[~train["battery_id"].isin(cal_cells)], train[train["battery_id"].isin(cal_cells)]
    res = np.abs(cal["rul"].to_numpy(float) - predict(name, proper, cal))
    n = len(res)
    q = np.quantile(res, min(1.0, np.ceil((n + 1) * (1 - ALPHA_CP)) / n), method="higher")
    ps = predict(name, proper, test)
    Rr, Mu = [], []
    for j in trc:
        tj, vj = train[train["battery_id"] != j], train[train["battery_id"] == j]
        r = np.abs(vj["rul"].to_numpy(float) - predict(name, tj, vj))
        mu = predict(name, tj, test)
        Rr.append(r); Mu.append(np.repeat(mu[None, :], len(r), axis=0))
    Rr, Mu = np.concatenate(Rr), np.concatenate(Mu, axis=0)
    n = len(Rr)
    lo_c = np.sort(Mu - Rr[:, None], axis=0)[max(int(np.floor(ALPHA_CP * (n + 1))) - 1, 0)]
    hi_c = np.sort(Mu + Rr[:, None], axis=0)[min(int(np.ceil((1 - ALPHA_CP) * (n + 1))) - 1, n - 1)]
    rows, bands = [], []
    full = predict(name, train, test)
    for meth, lo, hi in (("split2", ps - q, ps + q), ("cvplus", lo_c, hi_c)):
        cov = (yte >= lo) & (yte <= hi); hw = (hi - lo) / 2
        row = dict(model=name, method=meth, held_out=held, life=eol, coverage=cov.mean(),
                   halfwidth=hw.mean(), width_over_life=hw.mean() / eol)
        for l in LAMBDAS:
            i = int(np.argmin(np.abs(cyc - round(l * eol))))
            row[f"cov_l{l}"] = float(cov[i]); row[f"hw_l{l}"] = float(hw[i])
        rows.append(row)
        bands.append(pd.DataFrame(dict(model=name, method=meth, battery=held, cycle=cyc, rul_true=yte,
                                       rul_pred=full, lo=np.clip(lo, 0, None), hi=hi)))
    return rows, bands

## SOC track helpers
Coulomb-counted label referenced to the early-life capacity Q_ref (Eq. 1 of the paper),
every-fifth-cycle sample of six cells.

In [ ]:
SOC_CELLS = ["B0005", "B0006", "B0007", "B0018", "B0042", "B0054"]
GROUP24 = {"B0005", "B0006", "B0007", "B0018"}
Q_NOMINAL = 2.0      # rated capacity of the PCoE 18650 cells (Ah)


def soc_load_meta():
    m = pd.read_csv(ROOT / "data" / "metadata.csv")
    d = m[m["type"] == "discharge"].copy()
    d["Capacity"] = pd.to_numeric(d["Capacity"], errors="coerce")
    d = d.dropna(subset=["Capacity"]); d = d[d["Capacity"] > 0.1]
    meta, qrefs = {}, {}
    for b, g in d.groupby("battery_id"):
        g = g.sort_values("uid").reset_index(drop=True)
        g["cycle"] = np.arange(1, len(g) + 1)
        qref = g["Capacity"].iloc[:max(5, int(0.25 * len(g)))].max()
        qrefs[b] = dict(q_ref=float(qref), n_cycles=len(g), temp=float(g["ambient_temperature"].iloc[0]))
        caps = g["Capacity"].to_numpy(float)
        for i, r in enumerate(g.itertuples()):
            meta[(b, r.filename)] = dict(cycle=int(r.cycle), cap=float(r.Capacity),
                                         cap_prev=float(caps[i - 1]) if i > 0 else float(caps[0]),
                                         qref=float(qref), temp=float(r.ambient_temperature))
    return meta, qrefs


def backward_diff(y, t):
    dy = np.diff(y, prepend=y[0]); dt = np.diff(t, prepend=t[0])
    dt[dt == 0] = np.nan
    out = dy / dt; out[0] = 0.0
    return np.nan_to_num(out)


def soc_build_table(meta):
    rows = []
    for b in SOC_CELLS:
        for f in sorted((ORGANIZED / b / "discharge").glob("*.csv")):
            key = (b, f.name)
            if key not in meta:
                continue
            info = meta[key]
            if (info["cycle"] - 1) % 5:
                continue                                   # every fifth cycle: 1, 6, 11, ...
            df = pd.read_csv(f)
            if len(df) < 30 or "Time" not in df.columns:
                continue
            df = df[df["Voltage_measured"] > 0].reset_index(drop=True)
            if len(df) < 30:
                continue
            t = df["Time"].to_numpy(float); V = df["Voltage_measured"].to_numpy(float)
            I = df["Current_measured"].to_numpy(float); T = df["Temperature_measured"].to_numpy(float)
            q = np.cumsum(np.abs(I) * np.diff(t, prepend=t[0])) / 3600.0
            soc = np.clip(100.0 * (info["cap"] - q) / info["qref"], -5, 105)
            rows.append(pd.DataFrame({
                "battery_id": b, "cycle": info["cycle"], "temp_amb": info["temp"],
                "soh": 100 * info["cap"] / info["qref"], "soh_prev": 100 * info["cap_prev"] / info["qref"],
                "qref": info["qref"], "cap_prev": info["cap_prev"],
                "t": t, "V": V, "I": I, "T": T,
                "dVdt": np.gradient(V, t, edge_order=1), "dTdt": np.gradient(T, t, edge_order=1),
                "dVdt_b": backward_diff(V, t), "dTdt_b": backward_diff(T, t),
                "V_roll10": pd.Series(V).rolling(10, min_periods=1).mean().to_numpy(),
                "T_roll10": pd.Series(T).rolling(10, min_periods=1).mean().to_numpy(),
                "q_removed": q, "P": V * np.abs(I), "soc": soc}))
    if not rows:
        raise FileNotFoundError(f"no discharge files under {ORGANIZED}; set PCOE_ORGANIZED")
    return pd.concat(rows, ignore_index=True)


SOC_BASE = ["V", "I", "T", "V_roll10", "T_roll10", "P", "temp_amb"]
SOC_FEATSETS = {
    "A_paper_SOHk":     SOC_BASE + ["dVdt", "dTdt", "q_removed", "soh"],
    "B_SOHprev_causal": SOC_BASE + ["dVdt_b", "dTdt_b", "q_removed", "soh_prev"],
    "C_noSOH":          SOC_BASE + ["dVdt_b", "dTdt_b", "q_removed"],
    "D_noSOH_noQ":      SOC_BASE + ["dVdt_b", "dTdt_b"],
}


def soc_models():
    from sklearn.linear_model import LinearRegression, Ridge
    from sklearn.ensemble import RandomForestRegressor
    import xgboost as xgb
    import lightgbm as lgb
    return {
        "Linear": LinearRegression(),
        "Ridge": Ridge(alpha=1.0, random_state=SEED),
        "RandomForest": RandomForestRegressor(n_estimators=200, max_depth=12, random_state=SEED, n_jobs=-1),
        "XGBoost": xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.9,
                                    colsample_bytree=0.9, random_state=SEED, verbosity=0, n_jobs=4),
        "LightGBM": lgb.LGBMRegressor(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.9,
                                      colsample_bytree=0.9, random_state=SEED, verbose=-1, n_jobs=4),
    }